In [1]:
from dataAnalysis.DataAnalysis import DataAnalysis
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

data = pd.read_csv(r"extdata/sbcdata.csv", header=0)
data_analysis = DataAnalysis(data)

/home/dwalke/git/sbc_app/dataAnalysis/data/Filter.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['Label'] = self.data['Diagnosis']
/home/dwalke/git/sbc_app/dataAnalysis/data/Filter.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['Label'] = self.data['Diagnosis']
/home/dwalke/git/sbc_app/dataAnalysis/data/Filter.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cave

In [2]:
sorted_train_data = data_analysis.get_training_data().sort_values(by="Id").reset_index(drop = True)
train_df = sorted_train_data.loc[:sorted_train_data.shape[0]*.8,:]
val_df = sorted_train_data.loc[sorted_train_data.shape[0]*.8:,:]
test_df = data_analysis.get_testing_data()
gw_df = data_analysis.get_gw_testing_data()

In [3]:
import numpy as np
import torch
from dataAnalysis.Constants import FEATURES, LABEL_COLUMN_NAME

def convert_cat_feature(df):
    df = df.copy()
    df["SexCategory"] = (df["Sex"] == "W").astype(int)
    return df
    
def get_graph(df):
    edge_index = []
    df = convert_cat_feature(df)
    df = df.sort_values(by=["Id", "Time"]).reset_index(drop=True)
    
    ## group df by ids
    for identifier, group in df.groupby("Id"):
        offset = group.index[0]
        triu_matrix = np.triu((group.index.values + np.identity(1))[0])
        triu_exp_matrix = np.expand_dims(triu_matrix, axis=-1)
    
        idx_shape = group.index.shape[0]
        idx_matrix = np.ones((idx_shape, idx_shape)) * np.arange(idx_shape) + 1 + offset
        idx_matrix = np.transpose(idx_matrix)
        idx_exp_matrix = np.expand_dims(idx_matrix, axis=-1)
    
        unprocess_edges = np.concatenate((idx_exp_matrix, triu_exp_matrix), axis=-1)
        reshaped_unprocess_edges = np.reshape(unprocess_edges, (-1, 2))
        mask = (reshaped_unprocess_edges[:, 0] * reshaped_unprocess_edges[:, 1]) != 0
        edge_index.append((reshaped_unprocess_edges[mask] - 1).astype(np.int64))
    edge_index_torch = torch.from_numpy(np.concatenate(edge_index)).type(torch.long).transpose(0,1)
    features_torch = torch.from_numpy(df[FEATURES].values).type(torch.float)
    labels_torch = torch.from_numpy((df.sort_values(by=["Id", "Time"])[LABEL_COLUMN_NAME] == "Sepsis").values).type(torch.long)
    return features_torch, edge_index_torch, labels_torch

In [4]:
X_train_comp, edge_index_train_comp, y_train_comp = get_graph(sorted_train_data)
X_train, edge_index_train, y_train = get_graph(train_df)
X_val, edge_index_val, y_val = get_graph(val_df)
X_test, edge_index_test, y_test = get_graph(test_df)
X_gw, edge_index_gw, y_gw = get_graph(gw_df)

In [5]:
rev_edge_index_train_comp = torch.zeros_like(edge_index_train_comp)
rev_edge_index_train_comp[0,:] = edge_index_train_comp[1,:]
rev_edge_index_train_comp[1,:] = edge_index_train_comp[0,:]

rev_edge_index_train = torch.zeros_like(edge_index_train)
rev_edge_index_train[0,:] = edge_index_train[1,:]
rev_edge_index_train[1,:] = edge_index_train[0,:]

rev_edge_index_val = torch.zeros_like(edge_index_val)
rev_edge_index_val[0,:] = edge_index_val[1,:]
rev_edge_index_val[1,:] = edge_index_val[0,:]

rev_edge_index_test = torch.zeros_like(edge_index_test)
rev_edge_index_test[0,:] = edge_index_test[1,:]
rev_edge_index_test[1,:] = edge_index_test[0,:]

rev_edge_index_gw = torch.zeros_like(edge_index_gw)
rev_edge_index_gw[0,:] = edge_index_gw[1,:]
rev_edge_index_gw[1,:] = edge_index_gw[0,:]

In [6]:
from torch_geometric.utils import to_undirected

undir_edge_index_comp = to_undirected(edge_index_train_comp)
undir_edge_index_train = to_undirected(edge_index_train)
undir_edge_index_val = to_undirected(edge_index_val)
undir_edge_index_test = to_undirected(edge_index_test)
undir_edge_index_gw = to_undirected(edge_index_gw)

/home/dwalke/.local/lib/python3.10/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


In [7]:
from EnsembleFramework import Framework
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

def user_function(kwargs):
    return kwargs["updated_features"] - kwargs["mean_neighbors"]


In [8]:
def get_features(framework, X, edge_index):
    return framework.get_features(X, edge_index, torch.ones(X.shape[0]).type(torch.bool))

In [9]:
dir_sets = [("train_comp", X_train_comp, edge_index_train_comp, y_train_comp), ("train", X_train, edge_index_train, y_train), ("val", X_val, edge_index_val, y_val), ("test", X_test, edge_index_test, y_test),
       ("gw", X_gw, edge_index_gw, y_gw)]
dir_sets_dict = {dir_set[0]: (dir_set[1:]) for dir_set in dir_sets}
rev_dir_sets = [("train_comp", X_train_comp, rev_edge_index_train_comp, y_train_comp), ("train", X_train, rev_edge_index_train, y_train), ("val", X_val, rev_edge_index_val, y_val), ("test", X_test, rev_edge_index_test, y_test),
       ("gw", X_gw, rev_edge_index_gw, y_gw)]
rev_dir_sets_dict = {rev_dir_set[0]: (rev_dir_set[1:]) for rev_dir_set in rev_dir_sets}
undir_sets = [("train_comp", X_train_comp, undir_edge_index_comp, y_train_comp), ("train", X_train, undir_edge_index_train, y_train), ("val", X_val, undir_edge_index_val, y_val), ("test", X_test, undir_edge_index_test, y_test),
       ("gw", X_gw, undir_edge_index_gw, y_gw)]
undir_sets_dict = {undir_set[0]: (undir_set[1:]) for undir_set in undir_sets}

In [10]:
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
def test_auroc_and_auprc(framework, clf, X, edge_index,y):
    features = torch.cat(get_features(framework, X, edge_index), dim = 1)
    pred_proba = clf.predict_proba(features.cpu())[:,1]
    auroc = roc_auc_score(y, pred_proba)

    precision, recall, thresholds = precision_recall_curve(y, pred_proba)
    auprc = auc(recall, precision)
    return auroc, auprc


In [11]:
def eval_rev(framework, clf):
    eval_dict = dict()
    for name in rev_dir_sets_dict:
        X, edge_index,y = rev_dir_sets_dict[name]
        auroc, auprc = test_auroc_and_auprc(framework, clf, X, edge_index,y)
        eval_dict[name] = dict()
        eval_dict[name]["AUROC"] = np.round(auroc, 4)
        eval_dict[name]["AUPRC"] = np.round(auprc, 4)
    return eval_dict
        
def eval_dir(framework, clf):
    eval_dict = dict()
    for name in dir_sets_dict:
        X, edge_index,y = dir_sets_dict[name]
        auroc, auprc = test_auroc_and_auprc(framework,clf, X, edge_index,y)
        eval_dict[name] = dict()
        eval_dict[name]["AUROC"] = np.round(auroc, 4)
        eval_dict[name]["AUPRC"] = np.round(auprc, 4)
    return eval_dict

In [16]:
from hyperopt import fmin, tpe, hp,STATUS_OK, SparkTrials, space_eval 

class SparkTune():
    def __init__(self, clf,user_function,hops,attention_config, y_train, X_train, train_edge_index, y_val, X_val, val_edge_index, parallelism=1):
        self.clf = clf
        self.user_function = user_function
        self.hops = hops
        self.attention_config = attention_config
        self.parallelism = parallelism
        self.y_train = y_train
        self.X_train= X_train
        self.train_edge_index = train_edge_index

        self.y_val = y_val
        self.X_val= X_val
        self.val_edge_index = val_edge_index

        
        
    def objective(self, params):
        framework = Framework(user_functions=[self.user_function for i in self.hops], 
                     hops_list=self.hops,
                     clfs=[None for i in self.hops],
                     gpu_idx=0,
                     handle_nan=0.0,
                    attention_configs=[self.attention_config for i in self.hops])
        features_train = torch.cat(get_features(framework, self.X_train, self.train_edge_index), dim = 1)
        features_val = torch.cat(get_features(framework, self.X_val, self.val_edge_index), dim = 1)
            
        if "max_depth" in params:
            params["max_depth"] = int(params["max_depth"]) if params["max_depth"] is not None else params["max_depth"]
        if "max_leaf_nodes" in params:
            params["max_leaf_nodes"] = int(params["max_leaf_nodes"]) if params["max_leaf_nodes"] is not None else params["max_leaf_nodes"]
        if "max_features" in params:
            params["max_features"] = int(params["max_features"]) if params["max_features"] is not None else params["max_features"]
        model = self.clf(**params)
        
        model.fit(features_train.cpu(), self.y_train)
        
        y_pred_proba = model.predict_proba(features_val.cpu())[:, 1]
        score = roc_auc_score(self.y_val, y_pred_proba)
        return {'loss': -score, 'status': STATUS_OK}
    
    def search(self, space, max_evals):
        spark_trials = SparkTrials(parallelism = self.parallelism)
        best_params = fmin(self.objective, space, algo=tpe.suggest, trials=spark_trials, max_evals=max_evals, verbose = True)
        return space_eval(space, best_params)

In [30]:
from hyperopt import hp
import numpy as np

control_weight = y_train.sum() / y_train.numel()
control_weight_scaled = (y_train.sum()*3) / (y_train.numel()*2)

dtree_choices = {
    "criterion": ["gini", "entropy", "log_loss"],
    "splitter": ["best", "random"],
    "random_state": [42],
}

dtree_space = {
    **{key: hp.choice(key, value) for key, value in dtree_choices.items()},
    'max_depth': hp.choice('max_depth', [None, hp.quniform('max_depth_value', 3, 100, 2)]),
    'min_samples_split': hp.uniform('min_samples_split', 0.01, 0.2),
    'min_samples_leaf': hp.uniform('min_samples_leaf', 0.001, 0.1),
    'max_features': hp.quniform('max_features', 2, 14, q = 1),
    'class_weight': hp.choice('class_weight', ["balanced", None, {0: control_weight, 1: 1}, {0: control_weight_scaled, 1: 1}]),
    # 'max_leaf_nodes': hp.choice('max_leaf_nodes', [None, hp.quniform('max_leaf_nodes_value', 10, 100, 1)]),
}

In [31]:
edge_type_sets = {
    "dir": dir_sets_dict,
    "rev_dir": rev_dir_sets_dict,
    # "undir": undir_sets_dict,
}

In [32]:
from tqdm.notebook import tqdm
from sklearn.linear_model import LogisticRegression

res_dict = dict()
for edge_type_set_name in tqdm(edge_type_sets):
    best_val = 0
    
    res_dict[edge_type_set_name] = dict()
    print("Find best hyperparams")
    X_train, edge_index_train, y_train = edge_type_sets[edge_type_set_name]["train"]
    X_val, edge_index_val, y_val = edge_type_sets[edge_type_set_name]["val"]
    spark_tune = SparkTune(DecisionTreeClassifier,user_function,[0,1],None, y_train, X_train, edge_index_train, y_val, X_val, edge_index_val, 3)
    params = spark_tune.search(dtree_space,200)
    
    if "max_depth" in params:
            params["max_depth"] = int(params["max_depth"]) if params["max_depth"] is not None else params["max_depth"]
    if "max_features" in params:
            params["max_features"] = int(params["max_features"]) if params["max_features"] is not None else params["max_features"]
    if "max_leaf_nodes" in params:
            params["max_leaf_nodes"] = int(params["max_leaf_nodes"]) if params["max_leaf_nodes"] is not None else params["max_leaf_nodes"]
    
    framework = Framework(user_functions=[user_function,user_function], 
                     hops_list=[0, 1],
                     clfs=[_, _],
                     gpu_idx=0,
                     handle_nan=0.0,
                    attention_configs=[None, None])
    print("Retrain with best params")
    X_train_comp, edge_index_train_comp, y_train_comp = edge_type_sets[edge_type_set_name]["train_comp"]
    features_train = torch.cat(get_features(framework, X_train_comp, edge_index_train_comp), dim = 1)
    model = DecisionTreeClassifier(**params)
    model.fit(features_train.cpu(), y_train_comp)

    print("Evaluate")
    eval_dict = dict()
    for name in edge_type_sets[edge_type_set_name]:
        X, edge_index,y = edge_type_sets[edge_type_set_name][name]
        auroc, auprc = test_auroc_and_auprc(framework,model, X, edge_index,y)
        eval_dict[name] = dict()
        eval_dict[name]["AUROC"] = np.round(auroc, 4)
        eval_dict[name]["AUPRC"] = np.round(auprc, 4)
    res_dict[edge_type_set_name] = dict()
    res_dict[edge_type_set_name]["best_params"] = params
    res_dict[edge_type_set_name]["eval_dict"] = eval_dict

  0%|          | 0/2 [00:00<?, ?it/s]

Find best hyperparams

  0%|                                                                           | 0/200 [00:00<?, ?trial/s, best loss=?]


  0%|▏                                                | 1/200 [00:05<16:40,  5.03s/trial, best loss: -0.557143289085449]


  2%|▋                                               | 3/200 [00:07<06:43,  2.05s/trial, best loss: -0.8158203028549964]


  2%|▉                                               | 4/200 [00:09<06:40,  2.04s/trial, best loss: -0.8158203028549964]

...


 98%|██████████████████████████████████████████████▎| 197/200 [06:38<00:07,  2.44s/trial, best loss: -0.928177677550428]


100%|██████████████████████████████████████████████▊| 199/200 [06:40<00:01,  1.84s/trial, best loss: -0.928177677550428]


100%|███████████████████████████████████████████████| 200/200 [06:44<00:00,  2.02s/trial, best loss: -0.928177677550428]

Total Trials: 200: 200 succeeded, 0 failed, 0 cancelled.



Retrain with best params
Evaluate


In [33]:
import pandas as pd
for key in res_dict:
    print(key)
    print(pd.DataFrame(res_dict[key]["eval_dict"]))
    print(pd.DataFrame(res_dict[key]["eval_dict"]))
    print(res_dict[key]["best_params"])

dir
       train_comp   train     val    test      gw
AUROC      0.8712  0.8728  0.8653  0.8501  0.7923
AUPRC      0.1281  0.1291  0.1248  0.1307  0.1071
       train_comp   train     val    test      gw
AUROC      0.8712  0.8728  0.8653  0.8501  0.7923
AUPRC      0.1281  0.1291  0.1248  0.1307  0.1071
{'class_weight': None, 'criterion': 'entropy', 'max_depth': None, 'max_features': 9, 'min_samples_leaf': 0.017380114884534385, 'min_samples_split': 0.04015651476483255, 'random_state': 42, 'splitter': 'best'}
rev_dir
       train_comp   train     val    test      gw
AUROC      0.9418  0.9422  0.9403  0.9270  0.9254
AUPRC      0.2090  0.2096  0.2070  0.1972  0.1752
       train_comp   train     val    test      gw
AUROC      0.9418  0.9422  0.9403  0.9270  0.9254
AUPRC      0.2090  0.2096  0.2070  0.1972  0.1752
{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': None, 'max_features': 12, 'min_samples_leaf': 0.01042576059847148, 'min_samples_split': 0.03151553134496568, 'ran